# Strands Agents with Bedrock AgentCore Code Interpreter — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Code Interpreter to give your AI agent the ability to execute Python code dynamically — applied to financial services use cases.

## Overview

In this lab, you will:
- Use the default Code Interpreter to run financial calculations in a sandbox
- Analyze transaction data for fraud patterns
- Calculate portfolio risk metrics (VaR, sector concentration)
- Create a custom Code Interpreter with network access for live market data

## Why Code Interpreter for FSI?

Financial services require:
- **Dynamic calculations** — Risk models, stress tests, scenario analysis
- **Data analysis** — Fraud detection, anomaly identification
- **Secure execution** — Sandboxed environment for sensitive financial data
- **Audit trail** — Every calculation is traceable

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore pandas

In [5]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Default Code Interpreter — Financial Calculations

The default Code Interpreter runs Python in a **sandboxed environment** with no network access. Perfect for secure financial calculations.

Let's test it with a portfolio risk calculation:

In [9]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

# Initialize the AgentCore Code Interpreter (default: sandboxed, no network)
agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create agent with default Code Interpreter (sandboxed)
risk_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="""You are a quantitative analyst assistant. You write and execute Python code 
    to perform financial calculations. Keep responses concise.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

risk_agent("Calculate the future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years.")

<thinking> To calculate the future value of an investment with compound interest, I need to use the formula:
FV = P * (1 + r/n)^(n*t)
Where:
- FV is the future value of the investment/loan, including interest
- P is the principal investment amount (the initial deposit or loan amount)
- r is the annual interest rate (decimal)
- n is the number of times that interest is compounded per year
- t is the time the money is invested or borrowed for, in years

In this case:
- P = $2,000,000
- r = 4.8% annual rate = 0.048
- n = 12 (compounded monthly)
- t = 5 years

I will use the code interpreter to execute the Python code that calculates the future value using this formula. </thinking>

Tool #1: code_interpreter
<thinking> It seems there was a syntax error due to the incorrect use of line continuation characters in the code. I will correct the code by removing the unnecessary backslashes and re-run the calculation. </thinking> 
Tool #2: code_interpreter
<thinking> The calculation has been succ

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking> The calculation has been successfully completed. The future value of the $2,000,000 investment at a 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44. </thinking>\n\nThe future value of the investment is approximately $2,541,281.44.'}], 'metadata': {'usage': {'inputTokens': 4463, 'outputTokens': 83, 'totalTokens': 4546}, 'metrics': {'latencyMs': 1005, 'timeToFirstByteMs': 463}}}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'code_interpreter': ToolMetrics(tool={'toolUseId': 'tooluse_4lLSpt1TfOVtwBW1OxcaOF', 'name': 'code_interpreter', 'input': {'code_interpreter_input': {'action': {'type': 'executeCode', 'language': 'python', 'code': 'P = 2000000\nr = 0.048\nn = 12\nt = 5\nFV = P * (1 + r/n)**(n*t)\nprint(FV)'}}}}, call_count=2, success_count=2, error_count=0, total_time=1.8731400966644287)}, cycle_durations=[4.220679759979248, 1.6691031455993652, 1

## Part 2: Fraud Detection on Transaction Data

Now let's give the agent our synthetic transaction dataset and ask it to identify fraud patterns.

The dataset (`data/transactions.csv`) contains 25 transactions with several suspicious patterns:
- **Velocity attack** — Multiple high-value transactions within seconds
- **Geo-anomaly** — Transactions in different countries within minutes
- **Escalating amounts** — Progressively larger transactions (testing limits)
- **Unusual timing** — High-value transactions at 3am

In [10]:
# Load the transaction data so we can pass it to the agent
import pandas as pd

transactions_df = pd.read_csv("../data/transactions.csv")
print(f"Loaded {len(transactions_df)} transactions")
transactions_df.head()

Loaded 25 transactions


,transaction_id,timestamp,customer_id,amount,currency,merchant,category,location_city,location_country,card_type,is_online
0,TXN-001,2026-05-28 08:15:23,CUST-4421,12.5,AUD,Morning Brew Cafe,Food & Drink,Sydney,AU,debit,False
1,TXN-002,2026-05-28 08:17:45,CUST-4421,3200.0,AUD,TechWorld Electronics,Electronics,Lagos,NG,debit,True
2,TXN-003,2026-05-28 08:18:12,CUST-4421,2800.0,AUD,GiftCards Express,Gift Cards,Lagos,NG,debit,True
3,TXN-004,2026-05-28 08:19:01,CUST-4421,1500.0,AUD,Crypto Exchange XYZ,Financial Services,Moscow,RU,debit,True
4,TXN-005,2026-05-28 12:30:00,CUST-4421,15.8,AUD,Lunch Spot,Food & Drink,Sydney,AU,debit,False


In [11]:
# Pass the data as context and ask the agent to analyze it
transaction_data = transactions_df.to_csv(index=False)

risk_agent(f"""Here is a CSV of transaction data:
{transaction_data}

Write Python code to detect fraud. Check for:
1. Velocity: >2 transactions within 5 minutes for same customer
2. Geo-anomaly: transactions in different countries within 30 minutes
3. Amount: single transaction > 000 or escalating pattern
4. Timing: transactions between midnight and 5am

Output a table with columns: transaction_id, customer_id, amount, flag_reason, risk_score (1-10).
Only show flagged transactions. Sort by risk_score descending.""")

<thinking> To detect fraud based on the given criteria, I need to:
1. Read the CSV data into a pandas DataFrame.
2. Check for each criterion and flag transactions accordingly.
3. Assign a risk score to each flagged transaction.
4. Output a table with the required columns and sort by risk score.

I will use the code interpreter to execute the Python code that performs these steps. </thinking> 
Tool #3: code_interpreter
<thinking> It seems there was an error because pandas does not have a StringIO attribute. I should use the io.StringIO module instead to read the CSV data. I will correct the code and re-run the fraud detection. </thinking> 
Tool #4: code_interpreter
<thinking> The fraud detection has been successfully completed. The output table shows the flagged transactions with their respective flag reasons and risk scores, sorted by risk score in descending order. </thinking>

The flagged transactions are as follows:

| transaction_id | customer_id | amount | flag_reason | risk_score

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking> The fraud detection has been successfully completed. The output table shows the flagged transactions with their respective flag reasons and risk scores, sorted by risk score in descending order. </thinking>\n\nThe flagged transactions are as follows:\n\n| transaction_id | customer_id | amount | flag_reason | risk_score |\n| --- | --- | --- | --- | --- |\n| TXN-002 | CUST-4421 | 3200.00 | Geo-anomaly | 7 |\n| TXN-004 | CUST-4421 | 1500.00 | Geo-anomaly | 7 |\n| TXN-015 | CUST-2201 | 22.00 | Escalating Pattern | 6 |\n| TXN-023 | CUST-3310 | 200.00 | Escalating Pattern | 6 |\n| TXN-020 | CUST-5543 | 1000.00 | Escalating Pattern | 6 |\n| TXN-002 | CUST-4421 | 3200.00 | Escalating Pattern | 6 |\n| TXN-025 | CUST-4421 | 18.90 | Escalating Pattern | 6 |\n| TXN-019 | CUST-5543 | 500.00 | Escalating Pattern | 6 |\n| TXN-018 | CUST-5543 | 300.00 | Escalating Pattern | 6 |\n| TXN-011 | CUST-9156 | 9

## Part 3: Portfolio Risk Analysis (VaR)

Let's analyze a portfolio using Value at Risk (VaR) — a standard risk metric in financial services.

We'll use the portfolio data from `data/portfolio.csv`.

In [12]:
portfolio_df = pd.read_csv("../data/portfolio.csv")
print(f"Loaded {len(portfolio_df)} positions")
portfolio_df.head(10)

Loaded 15 positions


,client_id,client_name,asset_class,ticker,units,purchase_price,current_price,currency,weight_pct,sector
0,CLI-001,Vanguard Super Fund,Equity,CBA.AX,15000,95.2,112.45,AUD,18.5,Financials
1,CLI-001,Vanguard Super Fund,Equity,BHP.AX,12000,42.8,45.60,AUD,12.2,Materials
2,CLI-001,Vanguard Super Fund,Equity,CSL.AX,3000,280.0,295.50,AUD,9.8,Healthcare
3,CLI-001,Vanguard Super Fund,Equity,WBC.AX,20000,22.5,25.80,AUD,8.6,Financials
4,CLI-001,Vanguard Super Fund,Equity,NAB.AX,18000,28.9,32.10,AUD,7.9,Financials
5,CLI-001,Vanguard Super Fund,Fixed Income,GOVT.AX,50000,100.0,98.50,AUD,15.0,Government Bonds
6,CLI-001,Vanguard Super Fund,Fixed Income,IAF.AX,30000,100.0,101.20,AUD,10.5,Corporate Bonds
7,CLI-001,Vanguard Super Fund,International,VGS.AX,8000,85.0,98.20,AUD,10.8,Global Equity
8,CLI-001,Vanguard Super Fund,Cash,CASH,500000,1.0,1.00,AUD,6.7,Cash
9,CLI-002,Block Treasury,Equity,AAPL,5000,175.0,198.50,USD,22.0,Technology


In [ ]:
portfolio_data = portfolio_df.to_csv(index=False)

risk_agent(f"""Analyze this portfolio for risk metrics. Calculate:
1. Total portfolio value (current prices × units) for each client
2. Sector concentration — what % is in each sector? Flag if any sector > 30%
3. Unrealized P&L per position (current vs purchase price)
4. Asset class allocation (Equity vs Fixed Income vs Cash vs Other)

Present results as a clear summary with any risk warnings.

Portfolio data:
{portfolio_data}""")

<thinking> To analyze the portfolio for risk metrics, I need to:
1. Calculate the total portfolio value for each client.
2. Determine the sector concentration and flag if any sector exceeds 30%.
3. Calculate the unrealized P&L per position.
4. Determine the asset class allocation.

I will use the code interpreter to execute the Python code that performs these calculations and generates the required summary

## Part 4: Custom Code Interpreter with Network Access

The default Code Interpreter is sandboxed (no internet). For use cases that need live data (e.g., fetching real stock prices), we create a **custom Code Interpreter with network access**.

### Step 1: Initialize AgentCore Clients

In [ ]:
from bedrock_agentcore._utils import endpoints
import boto3

region = boto3.session.Session().region_name

data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client('bedrock-agentcore-control',
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client('bedrock-agentcore',
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

print(f'✅ AgentCore clients initialized (region: {region})')

### Step 2: Create Custom Code Interpreter with Network Access

In [ ]:
from botocore.exceptions import ClientError

interpreter_name = 'fsi-risk-analyzer'

try:
    interpreter_response = cp_client.create_code_interpreter(
        name=interpreter_name,
        description='FSI Code Interpreter with network access for live market data',
        networkConfiguration={'networkMode': 'PUBLIC'}
    )
    interpreter_id = interpreter_response['codeInterpreterId']
    print(f'✅ Created interpreter: {interpreter_id}')
except ClientError as e:
    if 'already exists' in str(e):
        for item in cp_client.list_code_interpreters()['codeInterpreterSummaries']:
            if item['name'] == interpreter_name:
                interpreter_id = item['codeInterpreterId']
                print(f'✅ Using existing interpreter: {interpreter_id}')
                break
    else:
        raise e

### Step 3: Create a Session and Test Live Data Access

In [ ]:
# Create a session in the custom code interpreter
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id
)
session_id = session_response['sessionId']
print(f'✅ Session created: {session_id}')

# Test: install yfinance and fetch a real stock price
result = dp_client.invoke_code_interpreter(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id,
    code='''
import subprocess
subprocess.run(['pip', 'install', '-q', 'yfinance'], capture_output=True)
import yfinance as yf
cba = yf.Ticker('CBA.AX')
info = cba.info
print(f"CBA.AX Live Price: {info.get('currentPrice', 'N/A')} AUD")
print(f"Market Cap: {info.get('marketCap', 0)/1e9:.1f}B AUD")
print(f"P/E Ratio: {info.get('trailingPE', 'N/A')}")
'''
)
print(result.get('output', result))

### Step 4: Use Custom Code Interpreter with Strands Agent

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def execute_python(code: str) -> str:
    '''Execute Python code in a secure sandbox with internet access.

    Args:
        code: Python code to execute
    '''
    result = dp_client.invoke_code_interpreter(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id,
        code=code
    )
    return result.get('output', str(result))

# Create agent with custom code interpreter
live_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a quantitative analyst with access to live market data.
    You can execute Python code to fetch real-time prices and calculate risk metrics.
    Use yfinance for market data. Keep responses concise.''',
    tools=[execute_python],
)

live_agent('Fetch the current prices of CBA.AX and WBC.AX and compare their P/E ratios.')

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {live_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta", max_width=60)
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan", max_width=40)
table.add_column("Tool Result", style="cyan", max_width=40)

for message in live_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], (text[-1][:200] + "...") if text and len(text[-1]) > 200 else (text[-1] if text else ""),
        tool_name[-1] if tool_name else "",
        (json.dumps(tool_input[-1])[:150] + "...") if tool_input else "",
        (json.dumps(tool_result[-1])[:150] + "...") if tool_result else ""
    )

console.print(table)

## Resource Cleanup (Optional)

Clean up the custom Code Interpreter to avoid charges:

In [ ]:
# Uncomment to clean up
# dp_client.stop_code_interpreter_session(
#     codeInterpreterIdentifier=interpreter_id,
#     sessionId=session_id
# )
# cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
# print('✅ Resources cleaned up')

## Summary

In this lab, you:

- ✅ Used the default Code Interpreter for secure financial calculations
- ✅ Analyzed transaction data for fraud patterns (velocity, geo-anomaly, timing)
- ✅ Calculated portfolio risk metrics (sector concentration, P&L, allocation)
- ✅ Created a custom Code Interpreter with network access for live market data
- ✅ Fetched real-time stock prices and compared bank P/E ratios

### FSI Takeaways

| Capability | FSI Application |
|-----------|----------------|
| Sandboxed execution | Secure risk calculations on sensitive data |
| Dynamic code generation | Ad-hoc analysis without pre-built reports |
| Network-enabled interpreter | Live market data, API integrations |
| Audit trail (agent loop) | Compliance — every calculation is traceable |

### Next: Lab 02 — Browser Automation
We'll use AgentCore Browser to monitor regulatory websites (APRA, ASX) and extract live financial data.